# Invariant Mass (with estimation)

In [1]:
from spyral.core.constants import AMU_2_MEV, QBRHO_2_P

from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.nuclear.target import GasTarget, load_target, SolidTarget
from spyral_utils.plot import Histogrammer

import polars as pl
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import vector
import lmfit

%matplotlib widget
plt.close()

nuclear_map = NuclearDataMap()
# Ion Chamber entrance and exit window, thickness in ug/cm^2
ic_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
# AT-TPC entrance window, thickness in ug/cm^2
attpc_window_material = SolidTarget(compound=[[1,1,14],[6,12,14],[7,14,4],[8,16,4]], thickness=1422.312, nuclear_data=nuclear_map)
# Ion Chamber gas material, pressure in Torr
ic_gas_material = GasTarget(compound=[(6,12,1),(9,18,4)], pressure=200.0, nuclear_data=nuclear_map)
ic_gas_thickness = 0.035 #m


In [2]:
nuclear_map = NuclearDataMap()

# Set some parameters
workspace_path_target = Path("/Volumes/research_EXT_2/spyral_target_alpha_width3")
workspace_path_decay = Path("/Volumes/research_EXT_2/spyral_decay_alpha_width3")

solver_result_path_target = workspace_path_target / "InterpSolver"
solver_result_path_decay = workspace_path_decay / "InterpSolver"
# solver_result_path = Path("/Volumes/researchEXT/O16/all_analysis_spyralv1.0/all_five_tracks_analysis/combined_5tracks_all") / "InterpSolver"
# Your AT-TPC target gas material
target_material_path = Path("/Users/pranjalsingh/Desktop/research_space_spyral/e20020_analysis/solver_gas_16O.json")

# Run number range (inclusive)
run_min = 0
run_max = 5
# Specify your nuclei

# The nucleus we observe (the one we fitted)
ejectile_z = 2
ejectile_a = 4

# The incoming nucleus (the beam)
projectile_z = 8
projectile_a = 16

fusion_z = 10
fusion_a = 20

# The target nucleus
target_z = 2
target_a = 4

# We calculate the residual for you
residual_z = target_z + projectile_z - ejectile_z
residual_a = target_a + projectile_a - ejectile_a


if residual_z < 0:
    raise Exception(f"Illegal nuclei! Residual Z: {residual_z}")  # noqa: TRY002
if residual_a < 1:
    raise Exception(f"Illegal nuclei! Residual A: {residual_a}")  # noqa: TRY002


In [3]:
target_material = load_target(target_material_path, nuclear_map)
if not isinstance(target_material, GasTarget):
    print('Target error!')

ejectile = nuclear_map.get_data(ejectile_z, ejectile_a)
projectile = nuclear_map.get_data(projectile_z, projectile_a)
fusion = nuclear_map.get_data(fusion_z, fusion_a)
target = nuclear_map.get_data(target_z, target_a)
residual = nuclear_map.get_data(residual_z, residual_a)
print(f"Reaction: {target}({projectile}, {ejectile}){residual}")
print(f"Target material: {target_material.ugly_string}")

# Initial beam energy
mass_amu = projectile.mass / AMU_2_MEV # If needed, to convert beam energy in MeV/u -> MeV
proj_energy_accel = 161.0 # MeV, the beam energy from the accelerator

# The beam energy after the ic entrance window
proj_energy_ic = proj_energy_accel #- ic_window_material.get_energy_loss(projectile, proj_energy_accel, np.array([0.0]))[0]
# The beam energy after the ic gas
proj_energy_ic_exit = proj_energy_ic #- ic_gas_material.get_energy_loss(projectile, proj_energy_ic, np.array([ic_gas_thickness]))[0]
# The beam energy after the ic exit window
proj_energy_post_ic = proj_energy_ic_exit #- ic_window_material.get_energy_loss(projectile, proj_energy_ic_exit, np.array([0.0]))[0]
# The beam energy after the AT-TPC entrace window
proj_energy_start = proj_energy_post_ic #- attpc_window_material.get_energy_loss(projectile, proj_energy_post_ic, np.array([0.0]))[0]
# The beam energy at the downstream end of the AT-TPC
proj_energy_stop = proj_energy_start - target_material.get_energy_loss(projectile, proj_energy_start, np.array([1.0]))[0] # Energy at far end of detector
print(f"Accelerator Beam energy: {proj_energy_accel} MeV")
print(f"Beam energy after IC (2 windows + gas): {proj_energy_post_ic} MeV")
print(f"Beam energy range in AT-TPC: {proj_energy_start}-{proj_energy_stop} MeV")


Reaction: 4He(16O, 4He)16O
Target material: (Gas)4He1
Accelerator Beam energy: 161.0 MeV
Beam energy after IC (2 windows + gas): 161.0 MeV
Beam energy range in AT-TPC: 161.0-110.84634259864373 MeV


In [4]:
# grammer = Histogrammer()
# grammer.add_hist1d('missing', 80, (5, 40.0))
# grammer.add_hist1d('inv', 80, (5, 40.0))
# grammer.add_hist1d('inv_20ne', 100, (20.0, 50.0))
# grammer.add_hist1d('scattering_largest',70,(0,90))
# grammer.add_hist1d('scattering_others',70,(0,90))
# grammer.add_hist1d("missing_correct", 70, (0,10e-5))
# grammer.add_hist1d("invariant_incorrect", 70, (0,10e-5))
# grammer.add_hist2d('correlation',(100,100),((10,27.5),(10,27.5)))

## Invariant $^{16}$ O

### Experimental

In [5]:
# target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

# counter_inv = 0
# counter_mmass = 0

# missing_mass_list = []
# invariant_mass_list = []

# for run in range(run_min, run_max + 1):
#     path = solver_result_path / f"run_{run:04d}_4He.parquet"
#     if not path.exists():
#         continue

#     df = pl.scan_parquet(path).collect()
#     df = df.with_row_index("row_id")
#     counts = df.group_by("event").len()
#     valid_events = counts.filter(pl.col("len") == 5).select(pl.col("event"))

#     df = df.join(valid_events, on="event", how="inner")
#     df
#     df_max_polar = (
#         df.sort("polar", descending=True)
#         .group_by("event")
#         .first()
#     )
    
#     df_no_max_polar = df.join(
#         df_max_polar.select("row_id"),
#         on="row_id",
#         how="anti"
#     )


#     common_events = (
#         set(df_max_polar["event"].to_numpy()) &
#         set(df_no_max_polar["event"].to_numpy())
#     )

#     df_max_polar = df_max_polar.filter(pl.col("event").is_in(list(common_events)))
#     df_no_max_polar = df_no_max_polar.filter(pl.col("event").is_in(list(common_events)))
#     vertices = df_max_polar.select(['vertex_x', 'vertex_y', 'vertex_z']).to_numpy()
#     distances = np.linalg.norm(vertices, axis=1)

#     projectile_ke = proj_energy_start - target_material.get_energy_loss(
#         projectile, proj_energy_start, distances
#     )

#     projectile_vector = vector.array({
#         "px": np.zeros(len(projectile_ke)),
#         "py": np.zeros(len(projectile_ke)),
#         "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
#         "E": projectile_ke + projectile.mass
#     })

#     brho = df_max_polar.select('brho').to_numpy().flatten()
#     momentum = brho * float(ejectile.Z) * QBRHO_2_P
#     polar = df_max_polar.select('polar').to_numpy().flatten()
#     az = df_max_polar.select('azimuthal').to_numpy().flatten()
    
#     polar_deg = polar * (180/np.pi)
    
#     ejectile_vector = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })

#     residual_vector = target_vector + projectile_vector - ejectile_vector
#     ex = residual_vector.mass - residual.mass
    
#     grammer.fill_hist1d('missing', ex)
#     grammer.fill_hist1d('scattering_largest',polar_deg)
#     counter_mmass += len(ex)
    
#     missing_mass_list.extend(ex)

#     invariant_energy = []

#     events_ls = df_no_max_polar.select("event").unique().to_numpy().flatten()

#     for event in events_ls:
#         df_event = df_no_max_polar.filter(pl.col("event") == event)
#         # if len(df_event) != 4:
#         #     continue
#         brho = df_event.select('brho').to_numpy().flatten()
#         momentum = brho * float(ejectile.Z) * QBRHO_2_P

#         energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#         total_energy = np.sum(energy)

#         polar = df_event.select('polar').to_numpy().flatten()
#         polar_deg_2 = polar * (180/np.pi)
        
#         grammer.fill_hist1d('scattering_others',polar_deg_2)
        
#         az = df_event.select('azimuthal').to_numpy().flatten()

#         px = momentum * np.sin(polar) * np.cos(az)
#         py = momentum * np.sin(polar) * np.sin(az)
#         pz = momentum * np.cos(polar)

#         px_tot = np.sum(px)
#         py_tot = np.sum(py)
#         pz_tot = np.sum(pz)

#         p_tot = np.sqrt(px_tot**2 + py_tot**2 + pz_tot**2)

#         inv_mass = np.sqrt(total_energy**2 - p_tot**2)
#         inv_energy = inv_mass - projectile.mass

#         invariant_energy.append(inv_energy)
        
#     inv_arr = np.array(invariant_energy)

#     # print("total inv computed:", len(inv_arr))
#     # print("nan count:", np.isnan(inv_arr).sum())
#     # print("min/max:", np.nanmin(inv_arr), np.nanmax(inv_arr))
#     invariant_energy_arr = np.array(invariant_energy)
#     invariant_mass_list.extend(invariant_energy_arr)
    
#     grammer.fill_hist1d('inv', invariant_energy_arr)
    
#     grammer.fill_hist2d('correlation',ex,invariant_energy_arr)
    
#     counter_inv += len(invariant_energy_arr)

# print("Missing mass:", counter_mmass)
# print("Invariant mass:", counter_inv)

# print("Missing mass energy length:", len(missing_mass_list))
# print("Invariant mass energy length:", len(invariant_mass_list))



### Simulation

In [6]:
# target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

# counter_inv = 0
# counter_mmass = 0

# missing_mass_list = []
# invariant_mass_list = []

# for run in range(run_min, run_max + 1):
#     path_decay = solver_result_path_decay / f"run_{run:04d}_4He.parquet" 
#     path_target = solver_result_path_target / f"run_{run:04d}_4He.parquet" 
    
#     if not path_decay.exists():
#         continue

#     df = pl.scan_parquet(path_decay).collect()
#     df_target = pl.scan_parquet(path_target).collect()
    
#     df = df.with_columns(
#     pl.col("event").cast(pl.Int64)
#     )

#     df_target = df_target.with_columns(
#         pl.col("event").cast(pl.Int64)
#     )

#     counts_decay = df.group_by("event").len()
#     counts_target = df_target.group_by("event").len()

#     valid_events_decay = (
#         counts_decay
#         .filter(pl.col("len") == 4)
#         .select("event")
#     )
    
#     valid_events_target = (
#         counts_target
#         .filter(pl.col("len") == 1)
#         .select("event")
#     )

#     df_decay = df.join(valid_events_decay, on="event", how="inner")
#     df_target = df_target.join(valid_events_target, on="event", how="inner")
    
    

#     events_decay = set(df_decay["event"].unique().to_list())
#     events_target = set(df_target["event"].unique().to_list())

#     common_events = events_decay & events_target

#     print("common events:", len(common_events))

#     df_max_polar = df_target.filter(
#         pl.col("event").is_in(common_events)
#     )

#     df_no_max_polar = df_decay.filter(
#         pl.col("event").is_in(common_events)
#     )

#     vertices = df_max_polar.select(['vertex_x', 'vertex_y', 'vertex_z']).to_numpy()
#     distances = np.linalg.norm(vertices, axis=1)

#     projectile_ke = proj_energy_start - target_material.get_energy_loss(
#         projectile, proj_energy_start, distances
#     )

#     projectile_vector = vector.array({
#         "px": np.zeros(len(projectile_ke)),
#         "py": np.zeros(len(projectile_ke)),
#         "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
#         "E": projectile_ke + projectile.mass
#     })

#     brho = df_max_polar.select('brho').to_numpy().flatten()
#     momentum = brho * float(ejectile.Z) * QBRHO_2_P
#     polar = df_max_polar.select('polar').to_numpy().flatten()
#     az = df_max_polar.select('azimuthal').to_numpy().flatten()
    
#     polar_deg = polar * (180/np.pi)
    
#     ejectile_vector = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })

#     residual_vector = target_vector + projectile_vector - ejectile_vector
#     ex = residual_vector.mass - residual.mass
    
#     grammer.fill_hist1d('missing', ex)
#     grammer.fill_hist1d('scattering_largest',polar_deg)
#     counter_mmass += len(ex)
    
#     missing_mass_list.extend(ex)

#     invariant_energy = []

#     events_ls = df_no_max_polar.select("event").unique().to_numpy().flatten()

#     for event in events_ls:
#         df_event = df_no_max_polar.filter(pl.col("event") == event)
#         # if len(df_event) !=cacaaaccccccc44 4:
#         #     continue
#         brho = df_event.select('brho').to_numpy().flatten()
#         momentum = brho * float(ejectile.Z) * QBRHO_2_P

#         energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#         total_energy = np.sum(energy)

#         polar = df_event.select('polar').to_numpy().flatten()
#         polar_deg_2 = polar * (180/np.pi)
        
#         grammer.fill_hist1d('scattering_others',polar_deg_2)
        
#         az = df_event.select('azimuthal').to_numpy().flatten()

#         px = momentum * np.sin(polar) * np.cos(az)
#         py = momentum * np.sin(polar) * np.sin(az)
#         pz = momentum * np.cos(polar)

#         px_tot = np.sum(px)
#         py_tot = np.sum(py)
#         pz_tot = np.sum(pz)

#         p_tot = np.sqrt(px_tot**2 + py_tot**2 + pz_tot**2)

#         inv_mass = np.sqrt(total_energy**2 - p_tot**2)
#         inv_energy = inv_mass - projectile.mass
        
#         grammer.fill_hist1d("missing_correct",ex)
#         chisq = df_event.select("redchisq").to_numpy().flatten()
            
#         invariant_energy.append(inv_energy)
        
#     inv_arr = np.array(invariant_energy)

#     # print("total inv computed:", len(inv_arr))
#     # print("nan count:", np.isnan(inv_arr).sum())
#     # print("min/max:", np.nanmin(inv_arr), np.nanmax(inv_arr))
#     invariant_energy_arr = np.array(invariant_energy)
#     invariant_mass_list.extend(invariant_energy_arr)
    
    
#     print(len(ex),len(invariant_energy_arr))
    
#     grammer.fill_hist1d('inv', invariant_energy_arr)
    
#     grammer.fill_hist2d('correlation',ex,invariant_energy_arr)
#     # grammer.fill_hist2d('correlation',ex,correct_peak_energy_invariant)
    
#     counter_inv += len(invariant_energy_arr)

# print("Missing mass:", counter_mmass)
# print("Invariant mass:", counter_inv)

# print("Missing mass energy length:", len(missing_mass_list))
# print("Invariant mass energy length:", len(invariant_mass_list))



In [7]:
# target_vector = vector.array({
#     "px": [0.0],
#     "py": [0.0],
#     "pz": [0.0],
#     "E": [target.mass]
# })

# counter_inv = 0
# counter_mmass = 0

# missing_mass_list = []
# invariant_mass_list = []

# correct_peak = []
# incorrect_peak = []

# correct_peak_ex = []
# correct_peak_energy_invariant = []

# for run in range(run_min, run_max + 1):

#     path_decay = solver_result_path_decay / f"run_{run:04d}_4He.parquet"
#     path_target = solver_result_path_target / f"run_{run:04d}_4He.parquet"

#     if not path_decay.exists():
#         continue

#     df_decay = pl.scan_parquet(path_decay).collect()
#     df_target = pl.scan_parquet(path_target).collect()

#     df_decay = df_decay.with_columns(
#         pl.col("event").cast(pl.Int64)
#     )

#     df_target = df_target.with_columns(
#         pl.col("event").cast(pl.Int64)
#     )

#     # -----------------------------------------
#     # Keep only valid multiplicity events
#     # -----------------------------------------

#     counts_decay = df_decay.group_by("event").len()
#     counts_target = df_target.group_by("event").len()

#     valid_events_decay = (
#         counts_decay
#         .filter(pl.col("len") == 4)
#         .select("event")
#     )

#     valid_events_target = (
#         counts_target
#         .filter(pl.col("len") == 1)
#         .select("event")
#     )

#     df_decay = df_decay.join(valid_events_decay, on="event", how="inner")
#     df_target = df_target.join(valid_events_target, on="event", how="inner")

#     # -----------------------------------------
#     # Common events
#     # -----------------------------------------

#     events_decay = set(df_decay["event"].unique().to_list())
#     events_target = set(df_target["event"].unique().to_list())

#     common_events = events_decay & events_target

#     print("common events:", len(common_events))

#     df_max_polar = df_target.filter(
#         pl.col("event").is_in(common_events)
#     )

#     df_no_max_polar = df_decay.filter(
#         pl.col("event").is_in(common_events)
#     )
    

#     # -----------------------------------------
#     # Missing mass calculation
#     # -----------------------------------------

#     vertices = df_max_polar.select(
#         ['vertex_x', 'vertex_y', 'vertex_z']
#     ).to_numpy()


#     distances = np.linalg.norm(vertices, axis=1)

#     projectile_ke = (proj_energy_start - target_material.get_energy_loss(projectile,proj_energy_start,distances))

#     projectile_vector = vector.array({
#         "px": np.zeros(len(projectile_ke)),
#         "py": np.zeros(len(projectile_ke)),
#         "pz": np.sqrt(
#             projectile_ke * (
#                 projectile_ke + 2.0 * projectile.mass
#             )
#         ),
#         "E": projectile_ke + projectile.mass
#     })

#     brho = df_max_polar.select('brho').to_numpy().flatten()

#     momentum = brho * float(ejectile.Z) * QBRHO_2_P

#     polar = df_max_polar.select('polar').to_numpy().flatten()
#     az = df_max_polar.select('azimuthal').to_numpy().flatten()

#     polar_deg = polar * (180 / np.pi)

#     ejectile_vector = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })

#     residual_vector = (target_vector + projectile_vector - ejectile_vector)

#     ex = residual_vector.mass - residual.mass

#     # -----------------------------------------
#     # Map event -> ex
#     # -----------------------------------------

#     event_ex_map = dict(
#         zip(
#             df_max_polar["event"].to_list(),
#             ex
#         )
#     )

#     # -----------------------------------------
#     # Histograms
#     # -----------------------------------------

#     grammer.fill_hist1d('missing', ex)
#     grammer.fill_hist1d('scattering_largest', polar_deg)

#     counter_mmass += len(ex)

#     missing_mass_list.extend(ex)

#     # -----------------------------------------
#     # Invariant mass calculation
#     # -----------------------------------------

#     invariant_energy = []

#     events_ls = (
#         df_no_max_polar
#         .select("event")
#         .unique()
#         .to_numpy()
#         .flatten()
#     )
    
#     for event in events_ls[:1]:
#         print(event)
#         df_event = df_no_max_polar.filter(
#             pl.col("event") == event
#         )

#         print(df_event)
#         brho_decay = df_event.select('brho').to_numpy().flatten()

#         momentum_decay = (
#             brho_decay
#             * float(ejectile.Z)
#             * QBRHO_2_P
#         )

#         energy = np.sqrt(
#             momentum_decay**2.0 + ejectile.mass**2.0
#         )

#         total_energy = np.sum(energy)
#         print(total_energy)

#         polar = df_event.select('polar').to_numpy().flatten()

#         polar_deg_2 = polar * (180 / np.pi)

#         grammer.fill_hist1d(
#             'scattering_others',
#             polar_deg_2
#         )

#         az = df_event.select('azimuthal').to_numpy().flatten()
#         px = momentum_decay * np.sin(polar) * np.cos(az)
#         py = momentum_decay * np.sin(polar) * np.sin(az)
#         pz = momentum_decay * np.cos(polar)

#         px_tot = np.sum(px)
#         py_tot = np.sum(py)
#         pz_tot = np.sum(pz)

#         p_tot = np.sqrt(
#             px_tot**2
#             + py_tot**2
#             + pz_tot**2
#         )

#         inv_mass = np.sqrt(
#             total_energy**2 - p_tot**2
#         )

#         inv_energy = inv_mass - projectile.mass

#         invariant_energy.append(inv_energy)

#         # -------------------------------------
#         # Correct peak selection
#         # -------------------------------------

#         if 10 <= inv_energy <= 18:

#             correct_peak.append(event)

#             correct_peak_energy_invariant.append(
#                 inv_energy
#             )

#             # matching missing mass ex
#             correct_peak_ex.append(
#                 event_ex_map[event]
#             )

#         elif 18 < inv_energy <= 26:

#             incorrect_peak.append(event)

#     # -----------------------------------------
#     # Convert arrays
#     # -----------------------------------------

#     invariant_energy_arr = np.array(invariant_energy)

#     invariant_mass_list.extend(
#         invariant_energy_arr
#     )

#     print(len(ex), len(invariant_energy_arr))

#     # -----------------------------------------
#     # Fill invariant histogram
#     # -----------------------------------------

#     grammer.fill_hist1d(
#         'inv',
#         invariant_energy_arr
#     )

#     # -----------------------------------------
#     # Correlation plot ONLY for correct peaks
#     # -----------------------------------------

#     grammer.fill_hist2d(
#         'correlation',
#         np.array(correct_peak_ex),
#         np.array(correct_peak_energy_invariant)
#     )

#     counter_inv += len(invariant_energy_arr)

# # ---------------------------------------------
# # Final statistics
# # ---------------------------------------------

# print("Missing mass:", counter_mmass)
# print("Invariant mass:", counter_inv)

# print(
#     "Missing mass energy length:",
#     len(missing_mass_list)
# )

# print(
#     "Invariant mass energy length:",
#     len(invariant_mass_list)
# )

In [8]:
# # ---------------------------------------------
# # Correlation histogram plot
# # ---------------------------------------------

# corr = grammer.get_hist2d("correlation")

# fig, ax = plt.subplots(1, 1)

# mesh = ax.pcolormesh(
#     corr.x_bins,
#     corr.y_bins,
#     corr.counts.T,
#     shading="auto"
# )

# # ---------------------------------------------
# # Colorbar
# # ---------------------------------------------

# cbar = fig.colorbar(mesh, ax=ax)
# cbar.set_label("Counts")

# # ---------------------------------------------
# # Labels and title
# # ---------------------------------------------

# ax.set_xlabel("Missing Method Ex [MeV]")
# ax.set_ylabel("Invariant Method Ex [MeV]")

# ax.set_title(
#     "Missing vs Invariant Correlation\n"
#     "(Correct Peak Events Only)"
# )

# # ---------------------------------------------
# # Optional formatting
# # ---------------------------------------------

# ax.tick_params(
#     axis='both',
#     which='major',
#     labelsize=12
# )

# # Optional diagonal reference line
# xmin = corr.x_bins.min()
# xmax = corr.x_bins.max()

# ax.plot(
#     [xmin, xmax],
#     [xmin, xmax],
#     linestyle="--",
#     linewidth=1.5
# )

# # ---------------------------------------------
# # Figure sizing
# # ---------------------------------------------

# fig.set_figheight(6.0)
# fig.set_figwidth(9.0)

# plt.tight_layout()
# plt.show()

In [9]:
# print(np.array(correct_peak))

In [10]:
# run = 0
# path_decay = solver_result_path_decay / f"run_{run:04d}_4He.parquet"  

# df = pl.scan_parquet(path_decay).collect()

# df = df.with_columns(
# pl.col("event").cast(pl.Int64)
# )

# print(df.filter(pl.col("event") == 3680))

In [11]:
# print(np.array(incorrect_peak))

In [12]:
# run = 0

# print(df.filter(pl.col("event") == 2912))

## Invariant Mass  $^{20}$ Ne

In [13]:
# target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

# for run in range(run_min, run_max + 1):
#     path = solver_result_path / f"run_{run:04d}_4He.parquet"
#     if not path.exists():
#         continue

#     df = pl.scan_parquet(path).collect()
#     df = df.with_row_index("row_id")
#     counts = df.group_by("event").len()
#     valid_events = counts.filter(pl.col("len") == 5).select(pl.col("event"))
#     df = df.join(valid_events, on="event", how="inner")
    
#     invariant_energy = []

#     events_fus = df.select("event").unique().to_numpy().flatten()
    
#     for event in events_fus:
#         df_event = df.filter(pl.col("event") == event)
        
#         brho = df_event.select('brho').to_numpy().flatten()
        
#         momentum = brho * float(ejectile.Z) * QBRHO_2_P

#         energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0)
        
#         total_energy = np.sum(energy)

#         polar = df_event.select('polar').to_numpy().flatten()
#         az = df_event.select('azimuthal').to_numpy().flatten()
        
#         px = momentum * np.sin(polar) * np.cos(az)
#         py = momentum * np.sin(polar) * np.sin(az)
#         pz = momentum * np.cos(polar)

#         px_tot = np.sum(px)
#         py_tot = np.sum(py)
#         pz_tot = np.sum(pz)

#         p_tot = np.sqrt(px_tot**2 + py_tot**2 + pz_tot**2)

#         inv_mass = np.sqrt(total_energy**2 - p_tot**2)
#         inv_energy = inv_mass - fusion.mass

#         invariant_energy.append(inv_energy)

#     inv_arr = np.array(invariant_energy)

#     invariant_energy_arr = np.array(invariant_energy)
#     grammer.fill_hist1d('inv_20ne', invariant_energy_arr)



## Missing Mass

In [14]:
# counter = 0
# target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

# for run in range(run_min, run_max+1):
    
#     df = None
#     path = solver_result_path/ f"run_{run:04d}_4He.parquet"
#     # path = solver_result_path / f"run_{run:04d}_{ejectile.isotopic_symbol}.parquet"
#     if not path.exists():
#         continue
#     df_parq = pl.scan_parquet(path)
    
#     # Example of a filter on the dataset, here we filter out the edges of the detector in Z
#     # df = df.filter((pl.col("vertex_z") > 0.005) & (pl.col("vertex_z") < 0.995) & ((pl.col("polar") < np.deg2rad(88.0)) | (pl.col("polar") > np.deg2rad(92.0)))).collect()
    
#     df_polar_max = df_parq.collect()
    
#     df_max = (
#         df_polar_max
#         .sort("polar", descending = True)
#         .group_by("event")
#         .first()
#     )
    
#     # if "event" not in df.columns:
#     #     print(f"Runs that don't have any data: {run}")
#     #     continue

#     # counter+= len(np.unique(df["event"]))


#     # Construct the projectile vectors (beam)
#     vertices = df_max.select(['vertex_x', 'vertex_y', 'vertex_z']).to_numpy()/1000 #covert to m (estiamtion in mm)
#     distances = np.linalg.norm(vertices, axis=1)
#     projectile_ke = proj_energy_start - target_material.get_energy_loss(projectile, proj_energy_start, distances)
#     projectile_vector = vector.array({
#         "px": np.zeros(len(projectile_ke)),
#         "py": np.zeros(len(projectile_ke)),
#         "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
#         "E": projectile_ke + projectile.mass
#     })
    
#     # Construct the ejectile vectors (detected)
#     brho = df_max.select('brho').to_numpy().flatten()
#     # print("brho:",brho)
#     momentum = df_max.select('brho').to_numpy().flatten() * float(ejectile.Z) * QBRHO_2_P
#     kinetic_energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0) - ejectile.mass
#     polar = df_max.select('polar').to_numpy().flatten()
#     az = df_max.select('azimuthal').to_numpy().flatten()
#     # cs = df.select('redchisq').to_numpy().flatten()
#     ejectile_vector = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })

#     # Do the kinematics
#     residual_vector = target_vector + projectile_vector - ejectile_vector # type: ignore
#     ex = residual_vector.mass - residual.mass # Excitation energy is "extra" mass
#     # print("ex",ex)
#     counter += len(ex)
#     grammer.fill_hist1d('est', ex)
    
# print(counter)
    

In [15]:
# target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})
# counter = 0 
# for run in range(run_min, run_max+1):
    
#     df = None
#     # path = estimate_result_path_max / f"run_{run:04d}.parquet"
#     path = solver_result_path / f"run_{run:04d}_{ejectile.isotopic_symbol}.parquet"
#     if not path.exists():
#         continue
#     df = pl.scan_parquet(path)
    
#     # Example of a filter on the dataset, here we filter out the edges of the detector in Z
#     # df = df.filter((pl.col("vertex_z") > 0.005) & (pl.col("vertex_z") < 0.995) & ((pl.col("polar") < np.deg2rad(88.0)) | (pl.col("polar") > np.deg2rad(92.0)))).collect()
    
#     df = df.collect()
    
#     # if "event" not in df.columns:
#     #     print(f"Runs that don't have any data: {run}")
#     #     continue

#     counter+= len(np.unique(df["event"]))


#     # Construct the projectile vectors (beam)
#     vertices = df.select(['vertex_x', 'vertex_y', 'vertex_z']).to_numpy()/1000 #covert to m (estiamtion in mm)
#     distances = np.linalg.norm(vertices, axis=1)
#     projectile_ke = proj_energy_start - target_material.get_energy_loss(projectile, proj_energy_start, distances)
#     projectile_vector = vector.array({
#         "px": np.zeros(len(projectile_ke)),
#         "py": np.zeros(len(projectile_ke)),
#         "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
#         "E": projectile_ke + projectile.mass
#     })
    
#     # Construct the ejectile vectors (detected)
#     brho = df.select('brho').to_numpy().flatten()
#     print("brho:",brho)
#     momentum = df.select('brho').to_numpy().flatten() * float(ejectile.Z) * QBRHO_2_P
#     kinetic_energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0) - ejectile.mass
#     polar = df.select('polar').to_numpy().flatten()
#     az = df.select('azimuthal').to_numpy().flatten()
#     # cs = df.select('redchisq').to_numpy().flatten()
#     ejectile_vector = vector.array({
#         "px": momentum * np.sin(polar) * np.cos(az),
#         "py": momentum * np.sin(polar) * np.sin(az),
#         "pz": momentum * np.cos(polar),
#         "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
#     })

#     # Do the kinematics
#     residual_vector = target_vector + projectile_vector - ejectile_vector # type: ignore
#     ex = residual_vector.mass - residual.mass # Excitation energy is "extra" mass
#     # print("ex",ex)
#     grammer.fill_hist1d('solver', ex)
    

## $^{16}$ O Excitation Plot

In [16]:
# fig, ax = plt.subplots(1,1)
# hist = grammer.get_hist1d("missing")
# hist_2 = grammer.get_hist1d("inv")
# print(np.sum(hist.counts),np.sum(hist_2.counts))

# ax.stairs(hist.counts, edges=hist.bins, color="navy", label="solv_missing")
# ax.stairs(hist_2.counts, edges=hist_2.bins, color="crimson", label="solv_inv")
# ax.set_xlabel("Excitation Energy [MeV]")
# ax.set_ylabel("Counts")
# plt.legend()
# plt.title(r"$^{16}$O Excitation Experiment")
# fig.set_figwidth(8.0)
# fig.set_figheight(6.0)
# plt.show()

## $^{20}$ Ne Excitation Plot

In [17]:
# fig, ax = plt.subplots(1,1)
# # hist = grammer.get_hist1d("solver")
# hist_3 = grammer.get_hist1d("inv_20ne")

# ax.stairs(hist_3.counts, edges=hist_3.bins, color="navy", label="solv_missing")
# ax.axvline(x=36.92, color='crimson', linestyle='--', linewidth=2, label='Excitation Fusion = 36.92 MeV')
# ax.set_xlabel("Excitation Energy [MeV]")
# ax.set_ylabel("Counts")
# plt.legend()
# plt.title(r"$^{20}$Ne Fusion")
# fig.set_figwidth(8.0)
# fig.set_figheight(6.0)
# plt.show()

## Scattering Angle Plot

In [18]:
# fig, ax = plt.subplots(1,1)
# # hist = grammer.get_hist1d("solver")
# hist_4 = grammer.get_hist1d('scattering_largest')
# hist_5 = grammer.get_hist1d('scattering_others')

# ax.stairs(hist_4.counts, edges=hist_4.bins, color="navy", label="largest")
# ax.stairs(hist_5.counts, edges=hist_5.bins, color="crimson", label="other 4")
# ax.set_xlabel("Angle [deg]")
# ax.set_ylabel("Counts")
# plt.legend()
# plt.title("Scattering Angle")
# fig.set_figwidth(8.0)
# fig.set_figheight(6.0)
# plt.show()

## Missing Mass vs Inv Mass Correlation Plot

In [19]:
# corr = grammer.get_hist2d("correlation")
# fig, ax = plt.subplots(1,1)
# mesh = ax.pcolormesh(corr.x_bins, corr.y_bins, corr.counts)
# fig.colorbar(mesh, ax=ax)
# ax.set_xlabel("Missing Method")
# ax.set_ylabel("Invariant Method")
# xmin = corr.x_bins.min()
# xmax = corr.x_bins.max()

# ax.plot(
#     [xmin, xmax],
#     [xmin, xmax],
#     linestyle="--",
#     linewidth=1.5
# )

# fig.set_figheight(8.0)
# fig.set_figwidth(11.0)
# plt.show()


In [20]:

# fig, ax = plt.subplots(figsize=(8, 6))
# ax.scatter(missing_mass_list, invariant_mass_list, color='navy', marker='o')

# # Add Labels
# ax.set_title("Correlation Plot")
# ax.set_xlabel("Missing Mass")
# ax.set_ylabel("Invariant Mass")
# ax.set_xlim(0,30)
# ax.set_ylim(0,40)

In [21]:
# fig, ax = plt.subplots(1,1)

# hist = grammer.get_hist1d("solver")
# hist_chi = grammer.get_hist1d("chisq_correct")
# hist_chi_2 = grammer.get_hist1d("chisq_incorrect")


# print(min(hist_chi.bins),max(hist_chi.bins))


# ax.stairs(hist_chi.counts, edges=hist_chi.bins, color="navy", label="correct peak")
# ax.stairs(hist_chi_2.counts, edges=hist_chi_2.bins, color="crimson", label="incorrect peak")
# ax.set_xlabel("Chisq")
# ax.set_ylabel("Counts")
# plt.legend()
# # plt.title(r"$^{16}$O Excitation Simulation")
# fig.set_figwidth(8.0)
# fig.set_figheight(6.0)
# plt.show()

## METHOD 1: Checking the largest polar overlap w/ target alpha

In [22]:
# true_alpha = 0
# total_alpha = 0

# for run in range(run_min, run_max + 1):
    
#     #### separated df
#     path_decay = solver_result_path_decay / f"run_{run:04d}_4He.parquet" 
#     path_target = solver_result_path_target / f"run_{run:04d}_4He.parquet" 
    
#     if not path_decay.exists():
#         continue

#     df = pl.scan_parquet(path_decay).collect()
#     df_target = pl.scan_parquet(path_target).collect()
    
#     df = df.with_columns(
#     pl.col("event").cast(pl.Int64)
#     )

#     df_target = df_target.with_columns(
#         pl.col("event").cast(pl.Int64)
#     )

#     counts_decay = df.group_by("event").len()
#     counts_target = df_target.group_by("event").len()

#     valid_events_decay = (
#         counts_decay
#         .filter(pl.col("len") == 4)
#         .select("event")
#     )
    
#     valid_events_target = (
#         counts_target
#         .filter(pl.col("len") == 1)
#         .select("event")
#     )

#     df_decay = df.join(valid_events_decay, on="event", how="inner")
#     df_target = df_target.join(valid_events_target, on="event", how="inner")
    
#     events_decay = set(df_decay["event"].unique().to_list())
#     events_target = set(df_target["event"].unique().to_list()) 
    
#     common_events = events_decay & events_target
#     print(len(common_events))
#     print("common events:", len(common_events))

#     df_max_polar = df_target.filter(
#         pl.col("event").is_in(common_events)
#     )

#     df_no_max_polar = df_decay.filter(
#         pl.col("event").is_in(common_events)
#     )

#     events_ls = df_no_max_polar.select("event").unique().to_numpy().flatten()
#     total_alpha+=len(events_ls)
#     # print(events_ls)
#     for event in events_ls:
#         df_event = df_no_max_polar.filter(pl.col("event") == event)
#         decay_polar =  df_event["polar"].to_numpy()
#         target_polar = df_max_polar.filter(pl.col("event")== event).select("polar").item()
#         result = np.all(target_polar > decay_polar)
        
#         if result:
#             true_alpha+=1

In [23]:
# print(f"True target percentage (method 1): {(true_alpha/total_alpha)*100:.2f}%")
# print(f"Total events examined (method 1): {total_alpha}")

## METHOD 2: Smallest different missing vs inv 

In [24]:
def mising_mass(target_alpha,target_vector):
    """Function to calculate the missing mass energy from the target alpha

    Args:
        target_alpha (dict): _description_
        projectile_vector (array): _description_

    Returns:
        float: the missing mass energy
    """
    distance = np.sqrt(target_alpha["vertex_x"]**2 + target_alpha["vertex_y"]**2 + target_alpha["vertex_z"]**2)
    distances_ls = [distance,distance]
    
    projectile_ke = proj_energy_start - target_material.get_energy_loss(
        projectile, proj_energy_start, distances_ls
    )

    projectile_vector = vector.array({
        "px": [0,0],
        "py": [0,0],
        "pz": np.sqrt(projectile_ke * (projectile_ke + 2.0 * projectile.mass)),
        "E": projectile_ke + projectile.mass
    })
    
    brho = target_alpha["brho"]
    momentum = brho * float(ejectile.Z) * QBRHO_2_P
    polar = target_alpha["polar"]
    az = target_alpha["azimuthal"]
    
    ejectile_vector = vector.array({
        "px": momentum * np.sin(polar) * np.cos(az),
        "py": momentum * np.sin(polar) * np.sin(az),
        "pz": momentum * np.cos(polar),
        "E": np.sqrt(momentum**2.0 + ejectile.mass**2.0)
    })

    residual_vector = target_vector + projectile_vector[0] - ejectile_vector
    ex = residual_vector.mass - residual.mass
    
    return ex


In [25]:
def inv_mass(decay_alphas):
    """This fucntion calcualtes the invariant mass energy 

    Args:
        decay_alphas (list): list of dicts of decay alphas 

    Returns:
        float: invariant mass energy
    """
    total_E, px_tot, py_tot, pz_tot = 0,0,0,0
    
    for row in decay_alphas:
        brho = row["brho"]
        momentum = brho * float(ejectile.Z) * QBRHO_2_P
        
        polar = row["polar"]
        az = row["azimuthal"]
        
        energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0)
        
        total_E += energy
        px_tot += momentum * np.sin(polar) * np.cos(az)
        py_tot += momentum * np.sin(polar) * np.sin(az)
        pz_tot += momentum * np.cos(polar)
        
    p_tot = np.sqrt(px_tot**2 + py_tot**2 + pz_tot**2)
        
    inv_mass = np.sqrt(total_E**2 - p_tot**2)
    inv_energy = inv_mass - projectile.mass
    
    return inv_energy
        
        
    

In [26]:
true_alpha_mth2 = 0
total_alpha_mth2 = 0

target_vector = vector.array({"px": [0.0], "py": [0.0], "pz": [0.0], "E": [target.mass]})

for run in range(run_min, run_max + 1):
    
    #### separated df
    path_decay = solver_result_path_decay / f"run_{run:04d}_4He.parquet" 
    path_target = solver_result_path_target / f"run_{run:04d}_4He.parquet" 
    
    if not path_decay.exists():
        continue

    df = pl.scan_parquet(path_decay).collect()
    df_target = pl.scan_parquet(path_target).collect()
    
    df = df.with_columns(
    pl.col("event").cast(pl.Int64)
    )

    df_target = df_target.with_columns(
        pl.col("event").cast(pl.Int64)
    )

    counts_decay = df.group_by("event").len()
    counts_target = df_target.group_by("event").len()

    valid_events_decay = (
        counts_decay
        .filter(pl.col("len") == 4)
        .select("event")
    )
    
    valid_events_target = (
        counts_target
        .filter(pl.col("len") == 1)
        .select("event")
    )

    df_decay = df.join(valid_events_decay, on="event", how="inner")
    df_target = df_target.join(valid_events_target, on="event", how="inner")
    
    events_decay = set(df_decay["event"].unique().to_list())
    events_target = set(df_target["event"].unique().to_list()) 
    
    common_events = events_decay & events_target ### LIST OF COMMON EVENTS
    
    df_max_polar = df_target.filter(
            pl.col("event").is_in(common_events)
        )

    df_no_max_polar = df_decay.filter(
        pl.col("event").is_in(common_events)
    )
    
    events_ls = df_no_max_polar.select("event").unique().to_numpy().flatten()
    
    # print(common_events)
    for event in events_ls:
        # print(event)
        true_target = df_max_polar.filter(pl.col("event") == event).row(0, named=True)
        
        df_decay = df_no_max_polar.filter(pl.col("event") == event)
            
        decay_rows = list(df_decay.iter_rows(named=True))
        
        alphas = [true_target] + decay_rows
        
        best_diff = np.inf
        best_idx = None
        for i in range(5):
            target = alphas[i]
            decay = [alphas[j] for j in range(5) if j != i]
            
            
            missing_energy = mising_mass(target,target_vector)
            invariant_energy = inv_mass(decay)
            
            diff = abs(missing_energy - invariant_energy)
            # print("For combination",i, missing_energy,invariant_energy, diff)
            if diff < best_diff:
                best_diff = diff
                best_idx = i
            
        total_alpha_mth2+=1
        
        # print("Best index difference is: ", best_idx)
        print()
        if best_idx == 0:
            true_alpha_mth2+=1
    

In [27]:
print(f"True target percentage (method 2): {(true_alpha_mth2/total_alpha_mth2)*100:.2f}%")
print(f"Total events examined (method 2): {total_alpha_mth2}")

True target percentage (method 2): 27.73%
Total events examined (method 2): 559
